In [6]:
import faiss
import numpy as np
import polars as pl # 高速DataFrame库，用于数据处理与分析，速度更快，内存占用更低
from gensim.test.utils import common_texts
from gensim.models import Word2Vec
import pandas as pd

In [7]:
train=pl.read_parquet('./data/processData/train.parquet')
test=pl.read_parquet('./data/processData/test.parquet')

In [8]:
# pl.concat纵向合并，groupby按照session分组，agg聚合函数(不保留原维度，合并session)，pl.col选择列，alias重命名
# 为每个session合并aid组合成句子
sentences_df=pl.concat([train,test]).group_by('session').agg(pl.col('aid').alias('sentence'))

In [9]:
sentences_df

session,sentence
i32,list[i32]
7833616,"[35825, 384537]"
10668478,"[467738, 467738, … 306860]"
13530731,[1655359]
7122637,"[93334, 1526603, … 1113013]"
7172182,"[1852036, 1852036]"
…,…
2624731,"[1495281, 986465, … 444351]"
6998353,"[1548804, 1548804]"
2222452,"[1634052, 404011, … 537137]"


In [10]:
sentences=sentences_df['sentence'].to_list()

In [11]:
sentences

[[35825, 384537],
 [467738, 467738, 530377, 530377, 530377, 306860],
 [1655359],
 [93334, 1526603, 325672, 1526603, 1503321, 1503321, 1503321, 1113013],
 [1852036, 1852036],
 [873805, 1043508, 1043508, 1043508, 1339717, 1339717],
 [912412, 790251, 790251],
 [630953, 1787723, 1099332, 1099332, 1414545, 1089421],
 [1231393, 705548, 418679],
 [637493, 637493],
 [1639091, 364358, 364358, 364358, 364358],
 [107206, 799533, 260981, 843034, 1586720, 1612914],
 [1259296,
  58223,
  1285692,
  58223,
  1522691,
  58223,
  1748313,
  1089544,
  1748313,
  58223,
  351793],
 [852868, 1416643, 874429, 720731],
 [954951, 954951],
 [1500008,
  1500008,
  1500008,
  1500008,
  65335,
  168896,
  168896,
  1027307,
  1027307,
  1455602,
  1677290,
  1677290,
  1565615,
  1565615,
  546497,
  454124,
  454124,
  454124],
 [1269764, 1672347, 1089331, 1731785],
 [717482,
  1233411,
  31744,
  421628,
  1233411,
  22582,
  1233411,
  22582,
  22582,
  1520100,
  1736621,
  103890,
  184703,
  682024,
  31

In [12]:
import os
# 做embedding
model_path='./model/w2vec.model'
if os.path.exists(model_path):
    w2vec=Word2Vec.load(model_path)
else:
    # sentences训练语料，vector_size词向量维度，min_count此至少出现一次才会被训练，workers训练时使用的CPU线程数
    w2vec=Word2Vec(sentences=sentences,vector_size=32,min_count=1,workers=4)
    w2vec.save(model_path)

In [13]:
aids=w2vec.wv.index_to_key # 按照出现频率从高到低返回词的列表
aids

[1460571,
 485256,
 108125,
 29735,
 1733943,
 832192,
 184976,
 166037,
 554660,
 986164,
 231487,
 1502122,
 1603001,
 1236775,
 322370,
 332654,
 1196256,
 756588,
 959208,
 1083665,
 1022566,
 620545,
 95488,
 801774,
 247240,
 673407,
 1645990,
 1586171,
 508883,
 1116095,
 1294924,
 530377,
 811371,
 1604220,
 892871,
 152547,
 714524,
 102345,
 1531805,
 409620,
 670006,
 544144,
 1257293,
 1498443,
 1197632,
 199409,
 584027,
 819288,
 1796103,
 612920,
 1743151,
 496180,
 399315,
 636101,
 500609,
 33343,
 77440,
 1647563,
 1685214,
 632365,
 1581568,
 1462420,
 1043508,
 1182614,
 1264313,
 329725,
 861401,
 1658239,
 1497089,
 881286,
 326904,
 1111967,
 984459,
 137514,
 634452,
 794192,
 1006198,
 803928,
 1125638,
 11830,
 305158,
 1365988,
 721034,
 1406660,
 10964,
 331708,
 385065,
 1255910,
 1624436,
 1636724,
 1142000,
 190818,
 1419849,
 670066,
 159789,
 1338993,
 493104,
 884502,
 1629608,
 842590,
 450505,
 1610239,
 1052212,
 1551213,
 1383529,
 1116621,
 135997

In [14]:
aid2idx={aid:i for i,aid in enumerate(aids)}

# w2vec.wv[aid]可以检索到向量
vecs=[x for x in w2vec.wv.vectors] # 词的向量
d=w2vec.wv.vectors.shape[1]

In [15]:
vecs

[array([-0.525585  ,  0.6858952 ,  2.785785  ,  2.6208563 , -0.26525363,
         0.29787347,  1.7074137 ,  0.4824451 , -0.16727161, -0.43419465,
         1.2344017 ,  1.2684562 , -1.6646498 ,  0.064592  , -0.6414619 ,
        -1.1295671 ,  0.1785714 ,  1.0859946 , -2.2185166 , -0.7079547 ,
         1.7932314 ,  2.0947704 ,  1.0100808 , -0.7484385 ,  1.5825579 ,
        -0.02127593,  0.49788022, -0.04550292,  0.22336864, -0.7678172 ,
        -0.8846277 ,  0.2522119 ], dtype=float32),
 array([-1.1955168 , -0.43625513,  2.2897859 , -0.75819427, -0.18765607,
        -0.24413006,  2.1355798 , -0.06653831, -0.1620256 ,  0.25122198,
         0.69855255, -0.14366879, -1.1667327 , -0.7300846 , -0.20105253,
         0.33913103, -0.68079364,  2.225522  , -1.7297244 ,  1.4625105 ,
         1.5165799 ,  3.1381626 ,  1.4753877 ,  0.564124  ,  0.22922829,
        -2.506062  , -1.8901013 , -0.926073  ,  0.16240212, -0.31780228,
        -0.51577824, -0.18228772], dtype=float32),
 array([ 0.1262586 , -

In [20]:
w2vec.wv.index_to_key[0] # 词的列表

1460571

In [16]:
vecs=np.ascontiguousarray(vecs,dtype='float32')

In [17]:
type(vecs)

numpy.ndarray

In [18]:
import faiss
# 建立faiss索引,传入向量的维度
index=faiss.IndexFlatL2(d)
index.add(vecs)

In [19]:
import pandas as pd
import numpy as np
from collections import defaultdict

sample_sub=pd.read_csv('./data/rawData/sample_submission.csv')

session_types=['clicks','carts','orders']
test_session_AIDs=test.to_pandas().reset_index(drop=True).groupby('session')['aid'].apply(list)
test_session_types=test.to_pandas().reset_index(drop=True).groupby('session')['type'].apply(list)

In [22]:
labels=[]

type_weight_multipliers={0:1,1:6,2:3}
for AIDs,types in zip(test_session_AIDs,test_session_types):
    if len(AIDs)>=20:
        # base底数，endpoint是否包含右边界
        weights=np.logspace(0.1,1,len(AIDs),base=2,endpoint=True)-1
        aids_temp=defaultdict(lambda: 0)
        for aid,w,t in zip(AIDs,weights,types):
            aids_temp[aid]+=w*type_weight_multipliers[t]
        sorted_aids=[k for k,v in sorted(aids_temp.items(),key=lambda item:-item[1])]
        labels.append(sorted_aids[:20]) # 取前20个
    else:
        AIDs=list(dict.fromkeys(AIDs[::-1]))
        most_recent_AIS=AIDs[0]

        q_idx=aid2idx[most_recent_AIS]
        q_vec=w2vec.wv.vectors[q_idx].reshape(1,-1)

        # D是距离，I是queries邻近的k个向量的索引（基于建立时的顺序）
        D,I=index.search(q_vec,21)

        nns=[w2vec.wv.index_to_key[i] for i in I[0][1:]]
        labels.append((AIDs+nns)[:20])

KeyboardInterrupt: 

In [ ]:
labels_as_strings=[' '.join([str(l) for l in lls]) for lls in labels] # 将预测的结果按转换为用空格分开的字符串

predictions=pd.DataFrame(data={'session_type':test_session_AIDs.index,'labels':labels_as_strings})
prediction_dfs=[]

for st in session_types:
    modified_predictions=predictions.copy()
    modified_predictions.session_type=modified_predictions.session_type.astype('str')+f'_{st}'
    prediction_dfs.append(modified_predictions)

submission=pd.concat(prediction_dfs).reset_index(drop=True)
submission.to_csv('./save/w2vec_submission.csv',index=False)